# Review provided data

**Purpose:** turn the source demand and map files into tables of substations, transmission routes and power stations for the fixed-capacity interruption workflow.

**Before running:**

- use the repository `.venv` kernel;
- place the expected folders under `data/0-incoming/energy/provided`, or set `MU_STAR_DATA_ROOT` to another data folder with the same structure;
- leave source files and filenames unchanged.

**Expected source inputs:**

- `power_demand/Power Demand.xlsx`, containing monthly peak demand and annual demand by customer group;
- `substation/Substation.{shp,shx,dbf,prj}`, containing substation point locations;
- `power_transmission/PowerGrid.{shp,shx,dbf,prj}`, containing transmission route geometry;
- `generation_source/GenSource1.{shp,shx,dbf,prj}`, containing mapped generation point records;
- `generation_source/GenSource2.{shp,shx,dbf,prj}`, containing mapped generation area records;
- optional reference material such as `substation/network_map_2025.png`.

**What this notebook does:** reads the supported source files, writes cleaned tables under `data/1-processed/energy/provided`, counts the records found, snaps substations to the transmission routes, maps their coverage and extracts the demand information in the workbook.

**User instructions:** review the generated `generators.csv` against your knowledge of the system. It combines mapped station locations, nearest snapped substations, and installed capacities for clearly matched plants in the linked CEB Annual Report 2023-2024. Unmatched values remain blank and are omitted from the PyPSA network until a source is added. The notebook does not estimate output from mapped area or treat unnamed polygons as power stations.

**Common adjustments:** you can use different data, change plot styles etc. If a new source has different columns or file formats, update the reading code.

In [ ]:
from importlib import reload
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import mu_star_energy.intake as intake
import mu_star_energy.paths as project_paths

# Reload local package modules so a long-running notebook kernel does not keep
# paths from before the data folders were renamed.
reload(project_paths)
reload(intake)

REPO_ROOT = project_paths.repo_root()
INPUT_DIR_OVERRIDE = None  # Set to Path(...) only for a one-off source location.
INPUT_DIR = (
    Path(INPUT_DIR_OVERRIDE).expanduser().resolve()
    if INPUT_DIR_OVERRIDE is not None
    else project_paths.incoming_energy_dir() / "provided"
)
OUTPUT_DIR = project_paths.processed_energy_dir() / "provided"

print("Reading source data from:", INPUT_DIR)
print("Writing cleaned data to:", OUTPUT_DIR)
intake.validate_provided_inputs(INPUT_DIR)

prepared = intake.prepare_provided_data(INPUT_DIR, OUTPUT_DIR)
prepared

## Power-station and network records found

The generated `generators.csv` keeps the source location, assigns the nearest snapped substation, and adds report-backed installed capacity where the station name has a clear match. A zero marginal cost is an equal-dispatch proxy for VoLL topology tests, not an operating-cost estimate. Unmatched technical values remain blank and visible for review.

In [ ]:
substations = gpd.read_parquet(prepared.substations)
snapped_substations = gpd.read_parquet(prepared.snapped_substations)
snap_distances = pd.read_csv(prepared.substation_snap_distances)
routes = gpd.read_parquet(prepared.transmission_routes)
generation_points = gpd.read_parquet(prepared.generation_points)
generation_areas = gpd.read_parquet(prepared.generation_areas)
generators = pd.read_csv(prepared.generators)
service_weights = pd.read_csv(prepared.service_weights)
generator_map_data = generators.copy()
monthly_peak = pd.read_csv(prepared.monthly_peak_demand, index_col="year")
annual_demand = pd.read_csv(prepared.annual_sector_demand)

inventory = pd.Series({
    "substations found": len(substations),
    "mapped transmission routes found": len(routes),
    "mapped generation points found": len(generation_points),
    "mapped generation areas found": len(generation_areas),
    "named power-station records": len(generators),
    "records with maximum output": int(generator_map_data["output_capacity_mw"].notna().sum()),
    "records with running cost": int(generator_map_data["marginal_cost"].notna().sum()),
    "records with connected substation": int(generator_map_data["bus_id"].notna().sum()),
    "substation service-weight records": len(service_weights),
})
display(inventory.to_frame("count"))

display_generators = generator_map_data[
    ["generator_id", "name", "asset_type", "output_capacity_mw", "effective_capacity_mw", "capacity_unit", "marginal_cost", "marginal_cost_basis", "bus_id", "bus_assignment_distance_m", "capacity_source"]
].rename(columns={
    "generator_id": "power-station ID",
    "name": "name",
    "asset_type": "fuel or technology",
    "output_capacity_mw": "maximum output",
    "effective_capacity_mw": "reported effective output",
    "capacity_unit": "capacity unit",
    "marginal_cost": "running cost",
    "marginal_cost_basis": "cost basis",
    "bus_id": "connected substation ID",
    "bus_assignment_distance_m": "station-to-substation distance (m)",
    "capacity_source": "capacity source",
})
display(display_generators)
display(service_weights)

## Transmission voltage and rating fields

The route table includes columns for `v_nom_kv`, `capacity_mw` and `capacity_unit` so values can be carried through when they are present in source columns or explicit route labels. In the current `PowerGrid.shp`, only one route name includes a machine-readable voltage: `CEB 66 KV Line St Martin / Henrietta`. No route-level MW capacity labels are present in the shapefile, so `capacity_mw` stays blank.

The reference image `network_map_2025.png` also has a legend for `66kV TRANSMISSION`, `132kV TRANSMISSION OPERATING AT 66kV`, and `SUBSTATION 66/22kV`. Those plot labels are useful review context, but they are not read automatically into individual route rows.

## Substation alignment warning

Every substation is moved to the nearest point on the mapped transmission routes so the points can be used when the route geometry is split into network sections. The source coordinates are retained separately.

The table is sorted by movement distance. Rows above 75 m are highlighted because they deserve particular attention when the network is checked against the reference map or better CEB data. Snapping aligns locations only; it does not determine line ratings or create links across gaps.


In [ ]:
snap_table = snap_distances[
    [
        "bus_id",
        "snapped_route_id",
        "snapped_route_name",
        "snapped_route_part_id",
        "snap_distance_m",
        "original_lon",
        "original_lat",
        "snapped_lon",
        "snapped_lat",
    ]
].copy()
snap_table["warning"] = snap_table["snap_distance_m"].apply(
    lambda distance: "check alignment" if distance > 75 else ""
)
display(
    snap_table.rename(columns={
        "bus_id": "substation ID",
        "snapped_route_id": "nearest route",
        "snapped_route_name": "route name",
        "snapped_route_part_id": "nearest route part",
        "snap_distance_m": "movement distance (m)",
        "original_lon": "source longitude",
        "original_lat": "source latitude",
        "snapped_lon": "snapped longitude",
        "snapped_lat": "snapped latitude",
    }).style
    .format({
        "movement distance (m)": "{:.1f}",
        "source longitude": "{:.5f}",
        "source latitude": "{:.5f}",
        "snapped longitude": "{:.5f}",
        "snapped latitude": "{:.5f}",
    })
    .map(
        lambda value: "background-color: #fde68a" if value > 75 else "",
        subset=["movement distance (m)"],
    )
)


## Map of the assets

Power-station areas are shown as transparent shapes above the transmission routes. Named sites are also shown as markers because many source polygons are too small to see at island scale.

In [ ]:
category_colors = {
    "thermal": "#8e24aa",
    "hydro": "#16a34a",
    "solar": "#c026d3",
    "wind": "#2563eb",
    "substation": "#f97316",
    "unspecified": "#9ca3af",
}
category_markers = {"thermal": "s", "hydro": "^", "solar": "v", "wind": "x", "unspecified": "D"}

fig, ax = plt.subplots(figsize=(10, 10))
routes.plot(ax=ax, color="#dc2626", linewidth=1.3, alpha=0.75, zorder=2)
for category, group in generation_areas.groupby("category"):
    alpha = 0.18 if category == "unspecified" else 0.4
    group.plot(
        ax=ax,
        color=category_colors.get(category, "#777777"),
        edgecolor="black" if category != "unspecified" else "none",
        linewidth=0.3,
        alpha=alpha,
        zorder=4,
    )
snapped_substations.plot(ax=ax, color="#ff8c00", edgecolor="black", markersize=42, zorder=6)
for category, group in generator_map_data.groupby("asset_type"):
    ax.scatter(
        group["lon"],
        group["lat"],
        c=category_colors.get(category, "#777777"),
        marker=category_markers.get(category, "D"),
        s=95,
        edgecolors="black" if category != "wind" else None,
        linewidths=1,
        label=category,
        zorder=8,
    )
capacity_rows = generator_map_data.dropna(subset=["output_capacity_mw"])
for _, row in capacity_rows.iterrows():
    ax.annotate(
        f"{row['output_capacity_mw']:g} MW_e",
        (row["lon"], row["lat"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=6.5,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1},
        zorder=10,
    )
if capacity_rows.empty:
    ax.text(
        0.01,
        0.01,
        "Generation capacities not yet populated",
        transform=ax.transAxes,
        fontsize=8,
        bbox={"facecolor": "white", "edgecolor": "#9ca3af", "alpha": 0.9},
    )
ax.set_title("Transmission, substations and generation capacity")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend(title="Named generation", fontsize="small")
plt.show()

## Demand information in the workbook

The workbook provides observed monthly system peaks and annual electricity use by customer group. It does not provide the hour-by-hour demand at each substation that the interruption model ultimately needs.

In [ ]:
display(monthly_peak.tail(8).style.format("{:.1f}"))
display(
    annual_demand.pivot(index="year", columns="category", values="demand_gwh")
    .tail(8)
    .style.format("{:.1f}")
)

## How to interpret and update the data

- Keep source files unchanged under `data/0-incoming`.
- Clear notebook outputs before committing if they contain private data or large figures.